AttentionPatternDiffusion 構想。注意パターンを移植することで、探索をショートカットができるらしい…というところから、仕様を注意パターンに変えるような、推論を学習に転嫁するような、RL Result Model Diffusion に似た枠組みを構想した。

アイデアのログは↓。

http://jrf.cocolog-nifty.com/statuses/2026/07/post-4cdc0e.html

その PoC コードとして Gemini さんが提案してくれたのが、以下の最初のコードになる。

基本的には「大きいモデル(AttentionGenerator)がルールである \[0, 1] または \[1, 0] で表されるルール…\[0,1] ならば Shift Right、\[1,0] ならば Reverse …について、小さいモデル(PatchedTransformerLayer)が、数値例から学習し、そのアテンション(パターン)について、学習を行い、AttentionGenerator が \[0,1] または \[1,0] を取って生成したアテンションが、確かに PatchedTransformerLayer のアテンションになりうる」…というコードになっている。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# シーケンス長（今回は8トークン）
N = 8

# ==========================================
# 1. 生成器 (Attention Generator)
# ==========================================
class AttentionGenerator(nn.Module):
    def __init__(self, spec_dim=2, seq_len=8):
        super().__init__()
        self.seq_len = seq_len
        # 仕様(2次元)からアテンションマップ(64次元)を直接出力する
        self.mlp = nn.Sequential(
            nn.Linear(spec_dim, 16),
            nn.ReLU(),
            nn.Linear(16, seq_len * seq_len)
        )

    def forward(self, spec):
        # spec shape: (Batch, spec_dim)
        logits = self.mlp(spec) # (Batch, seq_len * seq_len)
        logits = logits.view(-1, self.seq_len, self.seq_len) # (Batch, seq_len, seq_len)
        # 各行に対してSoftmaxをかけ、有効なアテンション分布にする
        attn_weights = torch.softmax(logits, dim=-1)
        return attn_weights

# ==========================================
# 2. 実行器 (Patched Transformer Layer)
# ==========================================
class PatchedTransformerLayer(nn.Module):
    def __init__(self, d_model=8):
        super().__init__()
        # 内部でのQ, K計算はバイパス。Valueの線形変換のみ定義
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        nn.init.eye_(self.v_proj.weight) # 初期値はアイデンティティ（恒等写像）

    def forward(self, x, attn_weights):
        # x: (Batch, seq_len, d_model)
        # attn_weights: (Batch, seq_len, seq_len) -> 外部から直接パッチされたもの
        v = self.v_proj(x)
        # アテンションによるルーティングを実行 (Batch Matrix Multiplication)
        out = torch.bmm(attn_weights, v)
        return out

# ==========================================
# 3. データセットとターゲットの作成
# ==========================================
# 入力 X (各位置を明確に区別するため、8x8の単位行列を使用)
X = torch.eye(N).unsqueeze(0) # Shape: (1, 8, 8)

# 仕様ベクトル (ワンホット)
spec_rev = torch.tensor([[1.0, 0.0]])   # Task A: Reverse
spec_shift = torch.tensor([[0.0, 1.0]]) # Task B: Shift Right

# 教師信号 Y (期待されるアテンションマップそのもの)
# Task A: 完全に反転した単位行列 (逆対角)
Y_rev = torch.flip(torch.eye(N), dims=[0]).unsqueeze(0)

# Task B: 右シフトした単位行列
Y_shift = torch.zeros(N, N)
for i in range(N):
    Y_shift[i, (i-1)%N] = 1.0
Y_shift = Y_shift.unsqueeze(0)

# ==========================================
# 4. エンドツーエンドのトレーニング
# ==========================================
generator = AttentionGenerator(spec_dim=2, seq_len=N)
executor = PatchedTransformerLayer(d_model=N)

# 生成器と実行器の全パラメータを最適化
optimizer = optim.Adam(list(generator.parameters()) + list(executor.parameters()), lr=0.01)
criterion = nn.MSELoss()

print("--- トレーニング開始 ---")
for epoch in range(401):
    optimizer.zero_grad()

    # タスク A: Reverse
    attn_rev = generator(spec_rev)
    out_rev = executor(X, attn_rev)
    loss_rev = criterion(out_rev, Y_rev)

    # タスク B: Shift Right
    attn_shift = generator(spec_shift)
    out_shift = executor(X, attn_shift)
    loss_shift = criterion(out_shift, Y_shift)

    loss = loss_rev + loss_shift
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | Total Loss: {loss.item():.6f} (Rev: {loss_rev.item():.6f}, Shift: {loss_shift.item():.6f})")

# ==========================================
# 5. 推論テスト（アテンション移植の可視化）
# ==========================================
print("\n--- 学習完了後のアテンションマップ移植テスト ---")
with torch.no_grad():
    # 1. Reverse仕様を入力
    patched_attn_rev = generator(spec_rev).squeeze(0)
    print("\n[仕様: Reverse [1, 0] を与えた時のアテンション行列]")
    for row in patched_attn_rev:
        print(" ".join([f"{val:.2f}" for val in row]))

    # 2. Shift Right仕様を入力
    patched_attn_shift = generator(spec_shift).squeeze(0)
    print("\n[仕様: Shift Right [0, 1] を与えた時のアテンション行列]")
    for row in patched_attn_shift:
        print(" ".join([f"{val:.2f}" for val in row]))

--- トレーニング開始 ---
Epoch   0 | Total Loss: 0.212645 (Rev: 0.107294, Shift: 0.105351)
Epoch 100 | Total Loss: 0.000053 (Rev: 0.000018, Shift: 0.000035)
Epoch 200 | Total Loss: 0.000010 (Rev: 0.000005, Shift: 0.000005)
Epoch 300 | Total Loss: 0.000004 (Rev: 0.000002, Shift: 0.000002)
Epoch 400 | Total Loss: 0.000002 (Rev: 0.000001, Shift: 0.000001)

--- 学習完了後のアテンションマップ移植テスト ---

[仕様: Reverse [1, 0] を与えた時のアテンション行列]
0.01 0.01 0.00 0.01 0.01 0.01 0.01 0.96
0.07 0.02 0.01 0.01 0.01 0.02 0.85 0.01
0.00 0.08 0.00 0.01 0.01 0.88 0.00 0.01
0.01 0.01 0.11 0.01 0.85 0.00 0.01 0.00
0.00 0.00 0.01 0.97 0.01 0.00 0.01 0.00
0.01 0.01 0.89 0.01 0.05 0.01 0.01 0.01
0.01 0.89 0.01 0.02 0.01 0.06 0.00 0.01
0.85 0.02 0.01 0.01 0.01 0.01 0.08 0.01

[仕様: Shift Right [0, 1] を与えた時のアテンション行列]
0.01 0.00 0.00 0.00 0.01 0.01 0.01 0.96
0.85 0.02 0.01 0.01 0.01 0.01 0.08 0.01
0.01 0.89 0.01 0.02 0.01 0.06 0.00 0.01
0.01 0.01 0.89 0.01 0.05 0.01 0.01 0.01
0.00 0.00 0.00 0.97 0.01 0.00 0.01 0.00
0.00 0.00 0.12 0.00 0.85 

2番目のコードは、Gemini さんのコードに対し Claude さんがコピペだけしているので意味がないと批判して、実際 Diffusion っぽい「創発」のようなことが起きるか試したものとなる。s = \[1,2,3,5,6,7] というシードに対して、shift を学習し、s = 4 で「創発」が起きるか見るものになっている。これは「創発」は起きなかった。s = 4 は s = 5 のコピペになってしまっている。なかなか「創発」は難しいようだ。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

N = 8

class AttentionGenerator(nn.Module):
    def __init__(self, spec_dim=1, seq_len=8, hidden=32):
        super().__init__()
        self.seq_len = seq_len
        self.mlp = nn.Sequential(
            nn.Linear(spec_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, seq_len * seq_len)
        )

    def forward(self, spec):
        logits = self.mlp(spec)
        logits = logits.view(-1, self.seq_len, self.seq_len)
        return torch.softmax(logits, dim=-1)

class PatchedTransformerLayer(nn.Module):
    def __init__(self, d_model=8):
        super().__init__()
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        nn.init.eye_(self.v_proj.weight)

    def forward(self, x, attn_weights):
        v = self.v_proj(x)
        return torch.bmm(attn_weights, v)

def shift_target(s, N):
    # row i attends to position (i - s) mod N
    Y = torch.zeros(N, N)
    for i in range(N):
        Y[i, (i - s) % N] = 1.0
    return Y

X = torch.eye(N).unsqueeze(0)

# s=4 is deliberately withheld from training
train_s = [1, 2, 3, 5, 6, 7]
test_s = 4

def spec_of(s):
    return torch.tensor([[s / N]])  # normalize to [0,1)-ish range

generator = AttentionGenerator(spec_dim=1, seq_len=N)
executor = PatchedTransformerLayer(d_model=N)
optimizer = optim.Adam(list(generator.parameters()) + list(executor.parameters()), lr=0.01)
criterion = nn.MSELoss()

targets = {s: shift_target(s, N).unsqueeze(0) for s in train_s}

print("--- training on s in", train_s, "(s=4 withheld) ---")
for epoch in range(2001):
    optimizer.zero_grad()
    total_loss = 0.0
    for s in train_s:
        attn = generator(spec_of(s))
        out = executor(X, attn)
        loss = criterion(out, targets[s])
        total_loss = total_loss + loss
    total_loss.backward()
    optimizer.step()
    if epoch % 400 == 0:
        print(f"Epoch {epoch:4d} | Total Loss: {total_loss.item():.6f}")

print("\n--- checking each trained s: does argmax per row match the ideal shift? ---")
with torch.no_grad():
    for s in train_s + [test_s]:
        attn = generator(spec_of(s)).squeeze(0)
        pred_idx = attn.argmax(dim=-1).tolist()
        ideal_idx = [(i - s) % N for i in range(N)]
        match = sum(p == g for p, g in zip(pred_idx, ideal_idx))
        ent = -(attn * attn.clamp_min(1e-9).log()).sum(dim=-1).mean().item()
        tag = "  <-- HELD OUT (s=4)" if s == test_s else ""
        print(f"s={s}: argmax={pred_idx} ideal={ideal_idx} match={match}/{N} mean_row_entropy={ent:.4f}{tag}")

print("\n--- full attention matrix for the held-out s=4 ---")
with torch.no_grad():
    attn4 = generator(spec_of(test_s)).squeeze(0)
    for row in attn4:
        print(" ".join([f"{v:.2f}" for v in row]))

--- training on s in [1, 2, 3, 5, 6, 7] (s=4 withheld) ---
Epoch    0 | Total Loss: 0.659901
Epoch  400 | Total Loss: 0.000287
Epoch  800 | Total Loss: 0.000075
Epoch 1200 | Total Loss: 0.000034
Epoch 1600 | Total Loss: 0.000019
Epoch 2000 | Total Loss: 0.000012

--- checking each trained s: does argmax per row match the ideal shift? ---
s=1: argmax=[7, 0, 1, 2, 3, 4, 5, 6] ideal=[7, 0, 1, 2, 3, 4, 5, 6] match=8/8 mean_row_entropy=0.0374
s=2: argmax=[6, 7, 0, 1, 2, 3, 4, 5] ideal=[6, 7, 0, 1, 2, 3, 4, 5] match=8/8 mean_row_entropy=0.0558
s=3: argmax=[5, 6, 7, 0, 1, 2, 3, 4] ideal=[5, 6, 7, 0, 1, 2, 3, 4] match=8/8 mean_row_entropy=0.0498
s=5: argmax=[3, 4, 5, 6, 7, 0, 1, 2] ideal=[3, 4, 5, 6, 7, 0, 1, 2] match=8/8 mean_row_entropy=0.0505
s=6: argmax=[2, 3, 4, 5, 6, 7, 0, 1] ideal=[2, 3, 4, 5, 6, 7, 0, 1] match=8/8 mean_row_entropy=0.0578
s=7: argmax=[1, 2, 3, 4, 5, 6, 7, 0] ideal=[1, 2, 3, 4, 5, 6, 7, 0] match=8/8 mean_row_entropy=0.0382
s=4: argmax=[3, 4, 5, 6, 7, 0, 1, 2] ideal=[4, 5